In [65]:
import numpy as np
import tensorflow as tf
import keras

In [ ]:
x = tf.constant([
    [
        [1,2,3],
        [4,5,6],
        [7,8,9]
    ],
    [
        [1.2,2.2,3.2],
        [4.2,5.2,6.2],
        [7.2,8.2,9.2]
    ],
])

print(x, x.shape)

tf.Tensor(
[[[1.  2.  3. ]
  [4.  5.  6. ]
  [7.  8.  9. ]]

 [[1.2 2.2 3.2]
  [4.2 5.2 6.2]
  [7.2 8.2 9.2]]], shape=(2, 3, 3), dtype=float32) (2, 3, 3)


In [47]:
x[:,:,:]

<tf.Tensor: shape=(2, 3, 3), dtype=float32, numpy=
array([[[1. , 2. , 3. ],
        [4. , 5. , 6. ],
        [7. , 8. , 9. ]],

       [[1.2, 2.2, 3.2],
        [4.2, 5.2, 6.2],
        [7.2, 8.2, 9.2]]], dtype=float32)>

In [48]:
tf.transpose(x)

<tf.Tensor: shape=(3, 3, 2), dtype=float32, numpy=
array([[[1. , 1.2],
        [4. , 4.2],
        [7. , 7.2]],

       [[2. , 2.2],
        [5. , 5.2],
        [8. , 8.2]],

       [[3. , 3.2],
        [6. , 6.2],
        [9. , 9.2]]], dtype=float32)>

In [51]:
tf.reduce_max(x)

<tf.Tensor: shape=(), dtype=float32, numpy=9.2>

In [53]:
np.array(x)

array([[[1. , 2. , 3. ],
        [4. , 5. , 6. ],
        [7. , 8. , 9. ]],

       [[1.2, 2.2, 3.2],
        [4.2, 5.2, 6.2],
        [7.2, 8.2, 9.2]]], dtype=float32)

In [ ]:
tf.constant(1) + tf.constant(2.1) # error
tf.constant(1.1) + tf.constant(2.1, dtype=tf.float64) # error

<tf.Tensor: shape=(), dtype=float32, numpy=3.1999998>

In [62]:
y = tf.Variable([1,2,3]) # can be varied
y.assign(2 * y)
y[0].assign(5)

<tf.Variable 'UnreadVariable' shape=(3,) dtype=int32, numpy=array([5, 4, 6])>

In [ ]:
def huber_func(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1
    squared_error = tf.square(error) / 2
    linear_error = tf.abs(error) - 0.5
    return tf.where(is_small_error, squared_error, linear_error)

# then : model.compile(loss=huber_func, optimizer='nadam')
# when loading : model = keras.models.load_model("model.h5", custom_objects={'huber_func':huber_func})

# huber func with custom threshold for considering as small error :
def create_huber_func(thredshold=1.0):
    if thredshold is None or thredshold < 0:
        raise ValueError("Invalid threshold value, it should be larger than 0")
    def huber_func(y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) < thredshold
        squared_error = tf.square(error) / 2
        linear_error = tf.abs(error) - 0.5
        return tf.where(is_small_error, squared_error, linear_error)
    return huber_func

# when loading : model = keras.models.load_model("model.h5", custom_objects={'huber_func':create_huber_func(2.0)})

class HuberLoss(keras.losses.Loss):
    def __init__(self, threshold, **kwargs):
        self.threshold = threshold
        super.__init__(**kwargs)

    def call(self, y_true, y_pred):
        pass # code

    def get_config(self):
        base_config = super().get_config()
        return {"threshold":self.threshold, **base_config}
    
# when loading : model = keras.models.load_model("model.h5", custom_objects={'huber_func':HuberLoss})


In [ ]:
class HuberMetric(keras.metrics.Metric):
    def __init__(self, t, **kwargs):
        self.huber_func = create_huber_func(t)
        self.total = self.add_weight('total', initializer='zeros')
        self.count = self.add_weight('count', initializer='zeros')
        super().__init__(kwargs)

    def update_state(self, *args, **kwargs):
        # return super().update_state(*args, **kwargs)
        return
    
    def result(self):
        return self.total / self.count
    
    def get_config(self):
        pass

class CustomDense(keras.layers.Layer):
    def __init__(self, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.activation = keras.activations.get(activation)

    def build(self, batch_input_shape):
        self.kernel = self.add_weight( # connections weights matrix
            name="kernel",
            shape=[batch_input_shape[:-1], self.units],
            initializer='glorotnormal'
        )
        self.bias = self.add_weight(
            name='bias',
            shape=[self.units],
            initializer='zeros'
        )
        super().build(batch_input_shape)
    
    def call(self, X):
        return self.activation(X @ self.kernel + self.bias)
    
    def computer_output_shape(self, batch_input_shape):
        return tf.TensorShape(batch_input_shape.as_list()[:-1] + self.units)
    
    def get_config(self):
        base_config = super().get_config()
        return {
            **base_config,
            "units":self.units,
            "activation":self.activations.serialize(self.activation)
        }